# Faithfulness experiments (Colab)
**Runtime → Change runtime type → T4 GPU.** Run top to bottom. Results are written straight to Google Drive,
so if Colab disconnects, just re-run the same cells: finished work is reused, not recomputed.

In [ ]:
# 1. Get / update the code
import os
if not os.path.exists('/content/BTech-Project'):
    # evidence-contradiction code lives on amey75/Btech-Project, branch contradiction-intervention
    !git clone -b contradiction-intervention https://github.com/amey75/Btech-Project.git /content/BTech-Project
else:
    !git -C /content/BTech-Project pull
%cd /content/BTech-Project
!pip -q install transformers sentencepiece accelerate nltk
import nltk; nltk.download('wordnet', quiet=True)   # word substitutions for evidence contradiction

In [ ]:
# 2. Drive, checkpoint and where to save results
import os, sys
os.chdir('/content/BTech-Project')          # cell 1 must have run once in this runtime
sys.path.insert(0, '/content/BTech-Project/faithfulness')
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/nt8'                       # folder with config.json + weights
RESULTS    = '/content/drive/MyDrive/btp_results'               # everything is saved here
RUN_DIR    = f'{RESULTS}/t5_nt8_test'                            # one folder per model/run
!ls "$MODEL_PATH" 

In [ ]:
# 3. Sanity check (should print contradiction + a sensible explanation)
import os, sys; os.chdir('/content/BTech-Project'); sys.path.insert(0, '/content/BTech-Project/faithfulness')
from predictors import T5Predictor
pred = T5Predictor(MODEL_PATH)
pairs = [("A man getting a tattoo on his back.", "A woman is getting a tattoo."),
         ("Two dogs run on the grass.", "Animals are outside."),
         ("A girl reads a book.", "The girl is reading her favourite novel.")]
full, fast = pred.predict(pairs), pred.predict_labels(pairs)
for f, g in zip(full, fast):
    print(f["label"], {k: round(v, 3) for k, v in f["probs"].items()}, "|", f["explanation"])
    assert f["label"] == g["label"] and all(abs(f["probs"][k] - g["probs"][k]) < 1e-3 for k in f["probs"]), "fast != full"
print("fast label-only scoring matches full generation")
del pred

## 4. Build the counterfactual dataset (model-independent, ~10 s)
Human-highlighted evidence in all 9,824 e-SNLI test pairs → removal / isolation variants with matched random controls.
Also validates the explanation-stance rules on the human explanations.

In [ ]:
%cd /content/BTech-Project
!python faithfulness/build_dataset.py --split test --out_dir "$RESULTS/dataset"
!cat "$RESULTS/dataset/esnli_cf_gold_test_stats.json" | head -40

## 5. Run the experiment
Start with `--n 2000` (~15-25 min on a T4). When that works, remove `--n 2000` to run all 9,824 test pairs
into a new `RUN_DIR` (~1-1.5 h; resumable). Out of memory → add `--batch_size 16 --label_batch_size 64`.

In [ ]:
%cd /content/BTech-Project
!python faithfulness/run_experiment.py --model_path "$MODEL_PATH" --n 2000 --out_dir "$RUN_DIR" 

## 5b. Evidence contradiction (all four models)
Replaces ONE evidence word with a conflicting one (man → woman, pool → ocean, red → blue). An independent NLI model
(DeBERTa-v3-large, the *oracle*) gives the label each edited pair should now have; the model is scored on whether it follows.

**Pilot first** (cell below, ~5 min): 300 pairs with random nt8 into a separate folder. Open
`spotcheck_contradiction.csv` from that folder, and each team member labels ~50 rows (is the edit natural? which label is right?).
Then run the four full models. Each full run re-uses that model's existing folder, so only the new edits are computed
(~15–25 min per model; the oracle cache in `btp_results/oracle_cache/` is shared, so later models are faster).

In [ ]:
%cd /content/BTech-Project
# Pilot: 300 pairs, random nt8, separate folder
PILOT_DIR = f'{RESULTS}/pilot_contradiction_nt8'
!python faithfulness/run_experiment.py --model_path /content/drive/MyDrive/nt8 --n 300 --out_dir "$PILOT_DIR"
import pandas as pd
ed = pd.read_csv(f'{PILOT_DIR}/contradiction_edits.csv')
ed[['variant','op','target','substitute','edited_premise','edited_hypothesis','oracle_label','oracle_conf','model_pred_on_edit']].sample(20, random_state=0)

In [ ]:
%cd /content/BTech-Project
# Full runs: the same folders as the removal/isolation results, so cached predictions are re-used
RUNS = {
    'random_nt8':       ('/content/drive/MyDrive/nt8',             f'{RESULTS}/t5_nt8_test_full'),
    'fastvotek_nt8':    ('/content/drive/MyDrive/fastvotek_nt8',   f'{RESULTS}/t5_esnli_fastvotek_nt8_full'),
    'random_nt128':     ('/content/drive/MyDrive/random_nt128',    f'{RESULTS}/t5_esnli_random_nt128_full'),
    'fastvotek_nt128':  ('/content/drive/MyDrive/fastvotek_nt128', f'{RESULTS}/t5_esnli_fastvotek_nt128_full'),
}
for name, (ckpt, run_dir) in RUNS.items():
    print(f'\n=================== {name}')
    !python faithfulness/run_experiment.py --model_path "$ckpt" --out_dir "$run_dir"
    !python faithfulness/plots.py --run_dir "$run_dir"

In [ ]:
# Counterfactual correctness, all four models side by side
import json, pandas as pd
rows = []
for name, (_, run_dir) in RUNS.items():
    s = json.load(open(f'{run_dir}/summary.json'))
    cc = s['counterfactual_correctness']
    for k, d in cc['contradiction'].items():
        for g in ['all', 'entailment', 'neutral', 'contradiction']:
            x = d[g]
            rows.append(dict(model=name, variant=k, group=g, edits=x['n_edits'], scored=x['n_confident'],
                             should_change=x['oracle_changes_label']['mean'], follows=x['follows_when_should_change']['mean'],
                             stays=x['stays_when_should_stay']['mean']))
    print(name, 'oracle accuracy on original pairs:', round(s['oracle']['accuracy_vs_gold_on_original_pairs'], 3))
cf = pd.DataFrame(rows)
cf[cf.group == 'all'].pivot(index='variant', columns='model', values='follows').round(3)

In [ ]:
%cd /content/BTech-Project
# 6. Figures + tables
!python faithfulness/plots.py --run_dir "$RUN_DIR"
from IPython.display import Image, display, Markdown
import glob
for f in sorted(glob.glob(f'{RUN_DIR}/figures/*.png')):
    display(Markdown(f'**{os.path.basename(f)}**')); display(Image(f))

In [ ]:
# 7. Table view of the main result (cited minus control, with 95% CI)
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
t = pd.read_csv(f'{RUN_DIR}/tables/contrasts.csv')
t[['contrast','group','n','flip_diff','flip_lo','flip_hi','drop_diff']].round(3)

In [ ]:
# 8. Examples to read: explanation argues for a different label than predicted
df = pd.read_csv(f'{RUN_DIR}/per_example.csv')
cols = ['gold','pred','p_pred','premise','hypothesis','explanation','explanation_stance']
df[~df.consistent][cols].head(15)

In [ ]:
# 9. Examples: premise evidence removed but entailment/contradiction prediction did not move
mask = (df.pred != 'neutral') & (df['premise:remove_cited_pred'] == df.pred) & (df['premise:remove_cited_drop'] < 0.05)
df[mask][cols + ['cited_premise','premise:remove_cited_drop','premise:remove_random_drop']].head(15)